# Subject 01 Visual Brain MLP Encoder
Train a direct fMRI-to-CLIP MLP encoder and evaluate the predicted CLIP image embeddings.

# 1. Train + Eval MLP Encoder
Load fMRI responses, train the MLP encoder, and evaluate CLIP prediction quality.

## Setup
Mount Drive, import libraries, set paths, and define compact hyperparameters.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, glob, json, random, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

BASE = "/content"
DRIVE_PROJECT_DIR = f"{BASE}/drive/MyDrive/maxwell-braid"
INPUT_DIR = f"{DRIVE_PROJECT_DIR}/inputs"
BETA_DIR = f"{BASE}/subject01_visual_brain_responses"
CLIP_DIR = f"{BASE}/subject01_clip_image_embeddings"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/outputs/mlp_encoder_{time.strftime('%Y%m%d_%H%M%S')}"
SUBJECT = "subj01"

SEED = 0
VAL_FRAC = 0.05
BATCH_SIZE = 512
MLP_EPOCHS = 15
MLP_LR = 1e-4
MLP_HIDDEN_DIMS = (2048, 4096)
MLP_DROPOUT = 0.50
CLIP_MSE_WEIGHT = 1.0
CLIP_COS_WEIGHT = 0.0
CLIP_CONTRASTIVE_WEIGHT = 0.0
CLIP_TEMPERATURE = 0.07
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.makedirs(OUTPUT_DIR, exist_ok=True)

for name in ["subject01_visual_brain_responses", "subject01_clip_image_embeddings"]:
    dst = f"{BASE}/{name}"
    if not os.path.exists(dst):
        shutil.copytree(f"{INPUT_DIR}/{name}", dst)

print(f"device={DEVICE} output={OUTPUT_DIR}")

## Load Data
Load paired fMRI/CLIP tensors, hold out shared-1000 as final test, then split remaining images into train/validation.


In [ ]:
import urllib.request
from collections import defaultdict
from scipy.io import loadmat

def session_id(path):
    return int(re.findall(r"\d+", os.path.basename(path))[-1])

beta_sess = {session_id(p): p for p in glob.glob(f"{BETA_DIR}/{SUBJECT}_visualroi_session*.pt")}
clip_sess = {session_id(p): p for p in glob.glob(f"{CLIP_DIR}/{SUBJECT}_clip_embeds*.pt")}
sessions = sorted(set(beta_sess) & set(clip_sess))
assert sessions, "No matched fMRI/CLIP sessions found."

betas = torch.cat([torch.load(beta_sess[s], map_location="cpu").float() for s in tqdm(sessions, desc="load fMRI")])
clips = torch.cat([torch.load(clip_sess[s], map_location="cpu").float() for s in tqdm(sessions, desc="load CLIP")])
assert len(betas) == len(clips), (betas.shape, clips.shape)

EXP = f"{BASE}/nsd_expdesign.mat"
if not os.path.exists(EXP):
    urllib.request.urlretrieve("https://natural-scenes-dataset.s3.amazonaws.com/nsddata/experiments/nsd/nsd_expdesign.mat", EXP)

mat = loadmat(EXP)
masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1
subjectim = mat["subjectim"].astype(np.int64) - 1
imgbrick_ids = subjectim[int(SUBJECT[-2:]) - 1, masterordering]
shared_ids = set(mat["sharedix"].reshape(-1).astype(np.int64) - 1)

def image_id_for_concat_trial(gidx):
    session = sessions[int(gidx) // 750]
    offset = int(gidx) % 750
    return int(imgbrick_ids[(session - 1) * 750 + offset])

img_of = np.array([image_id_for_concat_trial(gidx) for gidx in range(len(betas))])
groups = defaultdict(list)
nonshared_groups = defaultdict(list)
for gidx, img_id in enumerate(img_of):
    if int(img_id) in shared_ids:
        groups[int(img_id)].append(gidx)
    else:
        nonshared_groups[int(img_id)].append(gidx)

shared_img_ids = list(groups)
shared_eval_betas = torch.stack([betas[idxs].mean(0) for idxs in groups.values()])
shared_eval_clips = torch.stack([clips[idxs[0]] for idxs in groups.values()])

g = torch.Generator().manual_seed(SEED)
is_shared = np.array([int(img_id) in shared_ids for img_id in img_of])
nonshared_img_ids = np.array(sorted(set(img_of[~is_shared])), dtype=np.int64)
image_perm = np.random.RandomState(SEED).permutation(nonshared_img_ids)
n_val_images = int(VAL_FRAC * len(image_perm))
val_img_ids = set(map(int, image_perm[:n_val_images]))
train_img_ids = set(map(int, image_perm[n_val_images:]))
in_val = np.array([int(img_id) in val_img_ids for img_id in img_of])
val_idx = torch.from_numpy(np.where(~is_shared & in_val)[0]).long()
train_idx = torch.from_numpy(np.where(~is_shared & ~in_val)[0]).long()
assert train_img_ids.isdisjoint(val_img_ids)
assert train_img_ids.isdisjoint(shared_ids) and val_img_ids.isdisjoint(shared_ids)

beta_mean = betas[train_idx].mean(0)
beta_std = betas[train_idx].std(0) + 1e-6
clip_mean = clips[train_idx].mean(0)
clip_std = clips[train_idx].std(0) + 1e-6
x = (betas - beta_mean) / beta_std
y_clip = (clips - clip_mean) / clip_std
shared_eval_x = (shared_eval_betas - beta_mean) / beta_std
shared_eval_y = (shared_eval_clips - clip_mean) / clip_std
INPUT_DIM = x.shape[1]
CLIP_DIM = y_clip.shape[1]

train_loader = DataLoader(TensorDataset(x[train_idx], y_clip[train_idx]), batch_size=BATCH_SIZE, shuffle=True, generator=g, pin_memory=True)
train_eval_loader = DataLoader(TensorDataset(x[train_idx], y_clip[train_idx]), batch_size=BATCH_SIZE, pin_memory=True)
val_loader = DataLoader(TensorDataset(x[val_idx], y_clip[val_idx]), batch_size=BATCH_SIZE, pin_memory=True)
test_loader = DataLoader(TensorDataset(shared_eval_x, shared_eval_y), batch_size=BATCH_SIZE, pin_memory=True)

print(f"sessions={sessions}")
print(f"betas={tuple(betas.shape)} clips={tuple(clips.shape)}")
print(f"shared_test_images={len(shared_img_ids)} shared_test_trials={sum(len(v) for v in groups.values())}")
print(f"train_images={len(train_img_ids)} val_images={len(val_img_ids)}")
print(f"train_trials={len(train_idx)} val_trials={len(val_idx)} test_images={len(shared_eval_x)}")


## MLP Encoder Architecture
Define a supervised fMRI-to-CLIP MLP encoder.

In [ ]:
class MLPEncoder(nn.Module):
    def __init__(self, input_dim, clip_dim, hidden_dims, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), nn.LayerNorm(hidden_dims[0]), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]), nn.LayerNorm(hidden_dims[1]), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dims[1], clip_dim),
        )

    def forward(self, x):
        return self.net(x)

def mlp_loss(pred_clip, true_clip):
    clip_mse = F.mse_loss(pred_clip, true_clip)
    clip_cos = 1 - F.cosine_similarity(pred_clip, true_clip, dim=1).mean()
    clip_contrast = torch.zeros((), device=pred_clip.device)
    loss = CLIP_MSE_WEIGHT * clip_mse + CLIP_COS_WEIGHT * clip_cos + CLIP_CONTRASTIVE_WEIGHT * clip_contrast
    return loss, clip_mse, clip_cos, clip_contrast

model = MLPEncoder(INPUT_DIM, CLIP_DIM, MLP_HIDDEN_DIMS, MLP_DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=MLP_LR)

## Train MLP Encoder
Train with the BRAIDv2 MSE-only CLIP regression objective.

In [ ]:
MLP_METRICS = ["loss", "clip_mse", "clip_cos", "clip_contrast"]

@torch.no_grad()
def evaluate_mlp(loader):
    model.eval()
    totals = np.zeros(len(MLP_METRICS))
    seen = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        parts = mlp_loss(model(xb), yb)
        totals += xb.size(0) * np.array([p.item() for p in parts])
        seen += xb.size(0)
    return dict(zip(MLP_METRICS, totals / seen))

mlp_history = []
for epoch in range(1, MLP_EPOCHS + 1):
    model.train()
    train_totals = np.zeros(len(MLP_METRICS))
    seen = 0
    pbar = tqdm(train_loader, desc=f"mlp {epoch:03d}/{MLP_EPOCHS}", leave=False)
    for xb, yb in pbar:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        parts = mlp_loss(model(xb), yb)
        loss = parts[0]
        loss.backward()
        optimizer.step()
        train_totals += xb.size(0) * np.array([p.item() for p in parts])
        seen += xb.size(0)
        pbar.set_postfix(loss=loss.item(), clip_cos=parts[2].item())

    row = {f"train_{k}": v for k, v in zip(MLP_METRICS, train_totals / seen)}
    row.update({f"val_{k}": v for k, v in evaluate_mlp(val_loader).items()})
    row["epoch"] = epoch
    mlp_history.append(row)
    tqdm.write(
        f"mlp_epoch={epoch:03d} train={row['train_loss']:.4f} "
        f"val={row['val_loss']:.4f} val_clip_cos={row['val_clip_cos']:.4f}"
    )

torch.save({
    "model": model.state_dict(),
    "beta_mean": beta_mean,
    "beta_std": beta_std,
    "clip_mean": clip_mean,
    "clip_std": clip_std,
}, f"{OUTPUT_DIR}/best_mlp_encoder.pt")

with open(f"{OUTPUT_DIR}/mlp_history.json", "w") as f:
    json.dump(mlp_history, f, indent=2)

## Evaluate MLP Encoder
Reload the best MLP encoder, report final split losses, and plot train vs validation loss.

In [ ]:
ckpt = torch.load(f"{OUTPUT_DIR}/best_mlp_encoder.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
final_train = evaluate_mlp(train_eval_loader)
final_val = evaluate_mlp(val_loader)
final_test = evaluate_mlp(test_loader)
print(f"final_train_loss={final_train['loss']:.6f}")
print(f"final_val_loss={final_val['loss']:.6f}")
print(f"final_test_loss={final_test['loss']:.6f}")
print(f"train_clip_mse={final_train['clip_mse']:.6f} val_clip_mse={final_val['clip_mse']:.6f} test_clip_mse={final_test['clip_mse']:.6f}")
print(f"train_clip_cos={final_train['clip_cos']:.6f} val_clip_cos={final_val['clip_cos']:.6f} test_clip_cos={final_test['clip_cos']:.6f}")

epochs = [h["epoch"] for h in mlp_history]
plt.plot(epochs, [h["train_loss"] for h in mlp_history], label="train total")
plt.plot(epochs, [h["val_loss"] for h in mlp_history], label="val total")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/mlp_train_val_loss.png", dpi=180, bbox_inches="tight")
plt.show()
print(f"saved={OUTPUT_DIR}/mlp_train_val_loss.png")

# 2. Final BRAID v2 Evaluation
Evaluate frozen MLP predictions on the shared-1000 set with BRAID-style embedding and image metrics.

## Shared-1000 Eval Set
Reuse the held-out shared-1000 test set for final BRAID v2 embedding and image evaluation.


In [ ]:
mlp_shared_loader = test_loader
print(f"shared_eval_images={len(shared_img_ids)} shared_betas={tuple(shared_eval_betas.shape)} shared_clips={tuple(shared_eval_clips.shape)}")
print("shared-1000 is the held-out test set; no shared-image trials are used for train/validation.")

## Final Frozen Evaluation
Collect shared-1000 raw MLP predictions and score them against actual CLIP embeddings.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown, HTML
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

model.eval()
for p in model.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def collect_mlp_outputs(loader, raw=False, desc="collect MLP predictions"):
    model.eval()
    preds, trues = [], []
    cm, cs = clip_mean.to(DEVICE), clip_std.to(DEVICE)
    for xb, y in tqdm(loader, desc=desc):
        xb = xb.to(DEVICE, non_blocking=True)
        true_norm = y.to(DEVICE, non_blocking=True)
        pred_norm = model(xb)
        if raw:
            preds.append((pred_norm * cs + cm).cpu())
            trues.append((true_norm * cs + cm).cpu())
        else:
            preds.append(pred_norm.cpu())
            trues.append(true_norm.cpu())
    return torch.cat(preds), torch.cat(trues)

def l2_normalize(a):
    return a / np.clip(np.linalg.norm(a, axis=1, keepdims=True), 1e-8, None)

def mean_center(a):
    return a - a.mean(axis=0, keepdims=True)

def pairwise_sq_dists(a, b):
    return np.clip((a ** 2).sum(1, keepdims=True) + (b ** 2).sum(1, keepdims=True).T - 2 * a @ b.T, 0, None)

def mmd_gaussian(a, b, max_n=1000):
    rng = np.random.RandomState(SEED)
    if len(a) > max_n:
        a = a[rng.choice(len(a), max_n, replace=False)]
    if len(b) > max_n:
        b = b[rng.choice(len(b), max_n, replace=False)]
    z = np.concatenate([a, b])
    sub = z[rng.choice(len(z), size=min(len(z), 500), replace=False)]
    d2 = pairwise_sq_dists(sub, sub)
    gamma = 1.0 / (2 * np.median(d2[d2 > 0]))
    kxx = np.exp(-gamma * pairwise_sq_dists(a, a))
    kyy = np.exp(-gamma * pairwise_sq_dists(b, b))
    kxy = np.exp(-gamma * pairwise_sq_dists(a, b))
    m, n = len(a), len(b)
    return float((kxx.sum() - np.trace(kxx)) / (m * (m - 1)) + (kyy.sum() - np.trace(kyy)) / (n * (n - 1)) - 2 * kxy.mean())

def c2st_scores(pred_np, true_np, max_n=2000):
    rng = np.random.RandomState(SEED)
    n = min(len(pred_np), len(true_np), max_n // 2)
    pi = rng.choice(len(pred_np), n, replace=False)
    ti = rng.choice(len(true_np), n, replace=False)
    out = {}
    for name, p, t in [
        ("raw", pred_np[pi], true_np[ti]),
        ("l2_normalized", l2_normalize(pred_np[pi]), l2_normalize(true_np[ti])),
        ("mean_centered", mean_center(pred_np[pi]), mean_center(true_np[ti])),
    ]:
        x_cls = np.concatenate([p, t])
        y_cls = np.array([0] * len(p) + [1] * len(t))
        out[f"C2ST_LogReg_{name}"] = float(cross_val_score(LogisticRegression(max_iter=1000), x_cls, y_cls, cv=5).mean())
        out[f"C2ST_RBFSVM_{name}"] = float(cross_val_score(SVC(kernel="rbf"), x_cls, y_cls, cv=5).mean())
    return out

def retrieval_top1(pred, true, n_loops=30, n_samples=300):
    rng = np.random.RandomState(SEED)
    pred = F.normalize(pred.to(DEVICE), dim=1)
    true = F.normalize(true.to(DEVICE), dim=1)
    fwd, bwd = [], []
    for _ in range(n_loops):
        idx = rng.choice(len(true), size=min(n_samples, len(true)), replace=False)
        idx = torch.tensor(idx, device=DEVICE)
        labels = torch.arange(len(idx), device=DEVICE)
        fwd.append(((pred[idx] @ true[idx].T).argmax(1) == labels).float().mean().item())
        bwd.append(((true[idx] @ pred[idx].T).argmax(1) == labels).float().mean().item())
    return float(np.mean(fwd)), stats.norm.interval(0.95, loc=np.mean(fwd), scale=np.std(fwd) / np.sqrt(n_loops)), float(np.mean(bwd)), stats.norm.interval(0.95, loc=np.mean(bwd), scale=np.std(bwd) / np.sqrt(n_loops))

def metric_rows(category, items):
    return [{"Category": category, "Metric": name, "Value": f"{value:.6f}"} for name, value in items]

def c2st_metric_rows(row):
    rows = []
    for view in ["raw", "l2_normalized", "mean_centered"]:
        rows.append({"Category": "C2ST", "Metric": f"{view} LogReg", "Value": f"{row[f'C2ST_LogReg_{view}']:.6f}"})
        rows.append({"Category": "C2ST", "Metric": f"{view} RBF-SVM", "Value": f"{row[f'C2ST_RBFSVM_{view}']:.6f}"})
    return rows

def display_report_table(rows, columns):
    df = pd.DataFrame(rows, columns=columns)
    html = df.to_html(index=False, escape=False, classes="eval-report-table", border=0)
    display(HTML("""
    <style>
      table.eval-report-table {
        border-collapse: collapse;
        margin: 8px 0 10px 0;
        min-width: 520px;
        font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
      }
      table.eval-report-table th {
        background: #384152;
        color: #ffffff;
        font-weight: 700;
        text-align: left;
        padding: 8px 12px;
        border: 1px solid #5b6475;
      }
      table.eval-report-table td {
        padding: 8px 12px;
        border: 1px solid #5b6475;
      }
      table.eval-report-table tr:nth-child(even) td {
        background: rgba(127, 127, 127, 0.10);
      }
    </style>
    """ + html))

def show_eval_report(row, title, fwd_ci=None, bwd_ci=None):
    display(Markdown(f"### {title}"))
    rows = []
    rows += metric_rows("Embedding", [("Embed MSE", row["EmbedMSE"]), ("Embed Cosine", row["EmbedCosine"]), ("MMD", row["MMD"])])
    rows += metric_rows("Retrieval", [("Forward Retrieval", row["FwdRetrieval"]), ("Backward Retrieval", row["BwdRetrieval"])])
    rows += metric_rows("Mixing", [("Mix Silhouette", row["MixSilhouette"]), ("Mix Domain Accuracy", row["MixDomainAcc"])])
    rows += c2st_metric_rows(row)
    image_keys = ["PixCorr", "SSIM", "AlexNet(2)", "AlexNet(5)", "InceptionV3", "CLIP", "EffNet-B", "SwAV"]
    if all(k in row for k in image_keys):
        rows += metric_rows("Image", [(k, row[k]) for k in image_keys])
    display_report_table(rows, ["Category", "Metric", "Value"])
    print(f"KPI Forward Retrieval: {row['FwdRetrieval']:.4f}" + (f" | 95% CI [{fwd_ci[0]:.4f}, {fwd_ci[1]:.4f}]" if fwd_ci is not None else ""))
    print(f"KPI Backward Retrieval: {row['BwdRetrieval']:.4f}" + (f" | 95% CI [{bwd_ci[0]:.4f}, {bwd_ci[1]:.4f}]" if bwd_ci is not None else ""))

pred_clip_raw, true_clip_raw = collect_mlp_outputs(mlp_shared_loader, raw=True, desc="collect final MLP predictions")
pred_np, true_np = pred_clip_raw.numpy(), true_clip_raw.numpy()
mix_n = min(len(pred_np), 1000)
mix_idx = np.random.RandomState(SEED).choice(len(pred_np), mix_n, replace=False)
mix_x = np.concatenate([pred_np[mix_idx], true_np[mix_idx]])
mix_y = np.array([0] * mix_n + [1] * mix_n)
fwd, fwd_ci, bwd, bwd_ci = retrieval_top1(pred_clip_raw, true_clip_raw)

final_eval = {
    "model": "mlp_encoder",
    "EmbedMSE": float(F.mse_loss(pred_clip_raw, true_clip_raw)),
    "EmbedCosine": float(F.cosine_similarity(pred_clip_raw, true_clip_raw, dim=1).mean()),
    "MixSilhouette": float(silhouette_score(mix_x, mix_y, metric="cosine")),
    "MixDomainAcc": float(cross_val_score(LogisticRegression(max_iter=1000), mix_x, mix_y, cv=5).mean()),
    "MMD": mmd_gaussian(pred_np, true_np),
    "FwdRetrieval": fwd,
    "BwdRetrieval": bwd,
}
final_eval.update(c2st_scores(pred_np, true_np))

eval_df = pd.DataFrame([final_eval]).set_index("model")
show_eval_report(final_eval, "Final Frozen MLP Encoder Eval", fwd_ci, bwd_ci)
eval_df.to_csv(f"{OUTPUT_DIR}/final_mlp_encoder_eval.csv")
torch.save({"pred_clip_raw": pred_clip_raw, "true_clip_raw": true_clip_raw, "shared_img_ids": shared_img_ids}, f"{OUTPUT_DIR}/final_clip_predictions.pt")
print(f"saved={OUTPUT_DIR}/final_mlp_encoder_eval.csv")

## Full Image Reconstruction Evals
Decode all shared-1000 predicted CLIP embeddings and compute the BRAID image-eval metrics.

In [ ]:
RUN_IMAGE_EVALS = True
IMAGE_DECODE_BATCH = 10
SHARED1000_CACHE_PATH = f"{INPUT_DIR}/shared1000_ground_truth_images_u8.pt"
LOCAL_SHARED1000_CACHE_PATH = f"{BASE}/shared1000_ground_truth_images_u8.pt"

if RUN_IMAGE_EVALS:
    if not os.path.exists(SHARED1000_CACHE_PATH) and not os.path.exists(LOCAL_SHARED1000_CACHE_PATH):
        raise FileNotFoundError(f"Missing shared-1000 cache: {SHARED1000_CACHE_PATH}")
    else:
        import sys, subprocess, scipy as sp

        def copy_with_progress(src, dst, desc, chunk_mb=64):
            total = os.path.getsize(src)
            tmp = f"{dst}.part"
            if os.path.exists(tmp):
                os.remove(tmp)
            with open(src, "rb") as fsrc, open(tmp, "wb") as fdst, tqdm(total=total, unit="B", unit_scale=True, desc=desc) as pbar:
                while True:
                    chunk = fsrc.read(chunk_mb * 1024 * 1024)
                    if not chunk:
                        break
                    fdst.write(chunk)
                    pbar.update(len(chunk))
            if os.path.getsize(tmp) != total:
                raise IOError(f"Incomplete copy: {tmp} has {os.path.getsize(tmp)} bytes, expected {total}")
            shutil.copystat(src, tmp)
            os.replace(tmp, dst)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "diffusers", "accelerate", "scikit-image"])
        from torchvision import transforms
        from torchvision.models.feature_extraction import create_feature_extractor
        from torchvision.models import alexnet, AlexNet_Weights, efficientnet_b1, EfficientNet_B1_Weights, inception_v3, Inception_V3_Weights
        from skimage.color import rgb2gray
        from skimage.metrics import structural_similarity as ssim_fn
        from diffusers import KandinskyV22Pipeline, KandinskyV22PriorPipeline

        if os.path.exists(LOCAL_SHARED1000_CACHE_PATH):
            all_images_u8 = torch.load(LOCAL_SHARED1000_CACHE_PATH, map_location="cpu")
        else:
            copy_with_progress(SHARED1000_CACHE_PATH, LOCAL_SHARED1000_CACHE_PATH, "copy shared-1000 cache")
            all_images_u8 = torch.load(LOCAL_SHARED1000_CACHE_PATH, map_location="cpu")
        all_images = all_images_u8.float().div(255)
        del all_images_u8
        print("all_images", tuple(all_images.shape))

        kandinsky_pbar = tqdm(total=3, desc="load Kandinsky", unit="stage")
        kandinsky_pbar.set_postfix_str("decoder")
        decoder = KandinskyV22Pipeline.from_pretrained("kandinsky-community/kandinsky-2-2-decoder", torch_dtype=torch.float16).to(DEVICE)
        kandinsky_pbar.update(1)
        kandinsky_pbar.set_postfix_str("prior")
        prior = KandinskyV22PriorPipeline.from_pretrained("kandinsky-community/kandinsky-2-2-prior", torch_dtype=torch.float16).to(DEVICE)
        kandinsky_pbar.update(1)
        kandinsky_pbar.set_postfix_str("negative embed")
        neg_embed = prior.get_zero_embed(1).to(DEVICE, torch.float16)
        kandinsky_pbar.update(1)
        kandinsky_pbar.close()

        def decode_predicted_images(batch_size=IMAGE_DECODE_BATCH):
            cached = f"{OUTPUT_DIR}/all_recons.pt"
            if os.path.exists(cached):
                print(f"loading cached reconstructions: {cached}")
                return torch.load(cached, map_location="cpu")
            embeds = pred_clip_raw.to(device=DEVICE, dtype=torch.float16)
            recons = []
            for start in tqdm(range(0, len(embeds), batch_size), desc=f"decoding {len(embeds)} images"):
                batch = embeds[start:start + batch_size]
                imgs = decoder(image_embeds=batch, negative_image_embeds=neg_embed.repeat(len(batch), 1), num_inference_steps=50, height=512, width=512).images
                recons.extend(transforms.ToTensor()(im) for im in imgs)
            all_recons = torch.stack(recons)
            torch.save(all_recons, cached)
            print(f"saved reconstructions: {cached} {tuple(all_recons.shape)}")
            return all_recons

        all_recons = decode_predicted_images()

        def eval_pixcorr(recons, images):
            resize = transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR)
            r = resize(images).reshape(len(images), -1).cpu().numpy()
            f = resize(recons).reshape(len(recons), -1).cpu().numpy()
            return float(np.mean([np.corrcoef(r[i], f[i])[0, 1] for i in range(len(r))]))

        def eval_ssim(recons, images):
            resize = transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR)
            img_gray = rgb2gray(resize(images).permute(0, 2, 3, 1).cpu().numpy())
            rec_gray = rgb2gray(resize(recons).permute(0, 2, 3, 1).cpu().numpy())
            scores = [ssim_fn(rec, im, data_range=1.0, gaussian_weights=True, sigma=1.5, use_sample_covariance=False) for rec, im in zip(rec_gray, img_gray)]
            return float(np.mean(scores))

        @torch.no_grad()
        def two_way_identification(recons, images, feature_model, preprocess, feature_layer=None):
            preds = feature_model(torch.stack([preprocess(r) for r in recons]).to(DEVICE))
            reals = feature_model(torch.stack([preprocess(im) for im in images]).to(DEVICE))
            if feature_layer is not None:
                preds, reals = preds[feature_layer], reals[feature_layer]
            preds = preds.float().flatten(1).cpu().numpy()
            reals = reals.float().flatten(1).cpu().numpy()
            r = np.corrcoef(reals, preds)[:len(images), len(images):]
            success = r < np.diag(r)
            return float(np.mean(np.sum(success, 0)) / (len(images) - 1))

        alex_model = create_feature_extractor(alexnet(weights=AlexNet_Weights.IMAGENET1K_V1), return_nodes=["features.4", "features.11"]).to(DEVICE).eval().requires_grad_(False)
        alex_preprocess = transforms.Compose([transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
        inception_model = create_feature_extractor(inception_v3(weights=Inception_V3_Weights.DEFAULT), return_nodes=["avgpool"]).to(DEVICE).eval().requires_grad_(False)
        inception_preprocess = transforms.Compose([transforms.Resize(342, interpolation=transforms.InterpolationMode.BILINEAR), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
        eff_model = create_feature_extractor(efficientnet_b1(weights=EfficientNet_B1_Weights.DEFAULT), return_nodes=["avgpool"]).to(DEVICE).eval().requires_grad_(False)
        eff_preprocess = transforms.Compose([transforms.Resize(255, interpolation=transforms.InterpolationMode.BILINEAR), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
        swav_model = torch.hub.load("facebookresearch/swav:main", "resnet50")
        swav_model = create_feature_extractor(swav_model, return_nodes=["avgpool"]).to(DEVICE).eval().requires_grad_(False)
        swav_preprocess = transforms.Compose([transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/openai/CLIP.git"])
        import clip as openai_clip
        clip_2way_model, _ = openai_clip.load("ViT-L/14", device=DEVICE)
        clip_2way_preprocess = transforms.Compose([transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR), transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])])

        def eval_effnet(recons, images):
            gt = eff_model(eff_preprocess(images).to(DEVICE))["avgpool"].reshape(len(images), -1).cpu().numpy()
            fk = eff_model(eff_preprocess(recons).to(DEVICE))["avgpool"].reshape(len(recons), -1).cpu().numpy()
            return float(np.mean([sp.spatial.distance.correlation(gt[i], fk[i]) for i in range(len(gt))]))

        def eval_swav(recons, images):
            gt = swav_model(swav_preprocess(images).to(DEVICE))["avgpool"].reshape(len(images), -1).cpu().numpy()
            fk = swav_model(swav_preprocess(recons).to(DEVICE))["avgpool"].reshape(len(recons), -1).cpu().numpy()
            return float(np.mean([sp.spatial.distance.correlation(gt[i], fk[i]) for i in range(len(gt))]))

        image_eval = dict(final_eval)
        image_eval.update({
            "PixCorr": eval_pixcorr(all_recons, all_images),
            "SSIM": eval_ssim(all_recons, all_images),
            "AlexNet(2)": two_way_identification(all_recons, all_images, alex_model, alex_preprocess, "features.4"),
            "AlexNet(5)": two_way_identification(all_recons, all_images, alex_model, alex_preprocess, "features.11"),
            "InceptionV3": two_way_identification(all_recons, all_images, inception_model, inception_preprocess, "avgpool"),
            "CLIP": two_way_identification(all_recons, all_images, clip_2way_model.encode_image, clip_2way_preprocess),
            "EffNet-B": eval_effnet(all_recons, all_images),
            "SwAV": eval_swav(all_recons, all_images),
        })
        image_eval_df = pd.DataFrame([image_eval]).set_index("model")
        show_eval_report(image_eval, "Final Full BRAID v2 Image Eval", fwd_ci, bwd_ci)
        image_eval_df.to_csv(f"{OUTPUT_DIR}/final_full_image_eval.csv")
        print(f"saved={OUTPUT_DIR}/final_full_image_eval.csv")



## Predicted Image Preview Grid
Display and save a large actual-vs-predicted image preview from cached decoded images, without rerunning decoding or metrics.

In [ ]:
PREVIEW_GRID_N = 48
PREVIEW_GRID_COLS = 6
PREVIEW_GRID_PATH = f"{OUTPUT_DIR}/final_reconstruction_preview_grid.png"
LOCAL_SHARED1000_CACHE_PATH = f"{BASE}/shared1000_ground_truth_images_u8.pt"
SHARED1000_CACHE_PATH = f"{INPUT_DIR}/shared1000_ground_truth_images_u8.pt"
RECONS_PATH = f"{OUTPUT_DIR}/all_recons.pt"
PREDICTIONS_PATH = f"{OUTPUT_DIR}/final_clip_predictions.pt"

assert os.path.exists(RECONS_PATH), f"missing reconstructions: {RECONS_PATH}"

if "all_recons" not in globals():
    all_recons = torch.load(RECONS_PATH, map_location="cpu")

if "all_images" not in globals():
    if os.path.exists(LOCAL_SHARED1000_CACHE_PATH):
        all_images_u8 = torch.load(LOCAL_SHARED1000_CACHE_PATH, map_location="cpu")
    else:
        all_images_u8 = torch.load(SHARED1000_CACHE_PATH, map_location="cpu")
    all_images = all_images_u8.float().div(255)
    del all_images_u8

if "shared_img_ids" not in globals() and os.path.exists(PREDICTIONS_PATH):
    shared_img_ids = torch.load(PREDICTIONS_PATH, map_location="cpu")["shared_img_ids"]

n_show = min(PREVIEW_GRID_N, len(all_recons))
rng = np.random.RandomState(SEED)
show_idx = rng.choice(len(all_recons), size=n_show, replace=False)
cols = min(PREVIEW_GRID_COLS, n_show)
rows = int(np.ceil(n_show / cols))

actual = F.interpolate(all_images[show_idx].float(), size=all_recons.shape[-2:], mode="bilinear", align_corners=False)
pred = all_recons[show_idx].float()
combined = torch.cat([actual, pred], dim=3).clamp(0, 1)

fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 3.0 * rows), constrained_layout=True)
axes = np.atleast_1d(axes).reshape(rows, cols)
for ax in axes.ravel():
    ax.axis("off")

for k, idx in enumerate(show_idx):
    ax = axes[k // cols, k % cols]
    ax.imshow(combined[k].permute(1, 2, 0))
    label = shared_img_ids[int(idx)] if "shared_img_ids" in globals() else int(idx)
    ax.set_title(f"shared id {label}\nactual | predicted", fontsize=9)
    ax.axis("off")

fig.suptitle("Actual NSD Images vs Predicted Reconstructions", fontsize=16)
plt.savefig(PREVIEW_GRID_PATH, dpi=180, bbox_inches="tight")
plt.show()
print(f"saved={PREVIEW_GRID_PATH}")